In [217]:
import glob
import os
import tqdm
from collections import defaultdict
import re
import numpy as np
import pandas as pd
from scipy.io import wavfile
from scipy import signal
import pandas as pd
from moviepy.video.io.ffmpeg_tools import ffmpeg_extract_subclip
import cv2
import matplotlib.pyplot as plt
import datetime
import scipy.signal as sps
import librosa

In [218]:
# Path to the nidaq data for a particular experiment
experiment_no = 421
trig_channels = [10,11] # headmount trigger channel


nidaq_folder = "D:/big_setup/experiment_{}/nidaq/".format(experiment_no)
video_folder = "D:/big_setup/experiment_{}/videos/".format(experiment_no)
exp_folder = "D:/big_setup/experiment_{}/".format(experiment_no)
cam_mic_sync_concat_folder = "D:/big_setup/experiment_{}/concatenated_data_cam_mic_sync/".format(experiment_no)
timestamp_file = "D:/big_setup/experiment_{}/camera_timestamps.csv".format(experiment_no)
camera_fps = 30

# Detection thresholds
# For external mic 0
ch_0_threshold_min = 1
headmic_118_threshold_min = 1
headmic_35_threshold_min = 1



In [219]:
ch_0_files = glob.glob(cam_mic_sync_concat_folder+"channel_00_file_*.wav")
headmic_35_files = glob.glob(cam_mic_sync_concat_folder+"headmic_35_file_*.wav")
headmic_118_files = glob.glob(cam_mic_sync_concat_folder+"headmic_118_file_*.wav")

In [220]:
# Find all the occurences of ticks
# For channel 0
add_sample_amt = 0
clicks_ch_0 = np.array([],dtype=np.uint64)
for idx,file in tqdm.tqdm(enumerate(ch_0_files)):
    sr,ch = wavfile.read(file)
    val = np.where(ch_0_threshold_min<ch)

    cleaned_detections_ch_0 = [val[0][0]]
    for j in val[0][1:]:
        if j-cleaned_detections_ch_0[-1] > 10000: # detections within 0.08 seconds are discareded to keep the first occurence only
            cleaned_detections_ch_0.append(j)
    clicks_ch_0 = np.append(clicks_ch_0,np.array(cleaned_detections_ch_0,dtype=np.uint64)+add_sample_amt)
    add_sample_amt+=len(ch)

20it [00:01, 16.89it/s]


In [221]:
# For channel headmic_118
add_sample_amt = 0
clicks_headmic_118 = np.array([],dtype=np.uint64)
for idx,file in tqdm.tqdm(enumerate(headmic_118_files)):
    sr,ch = wavfile.read(file)
    val = np.where(headmic_118_threshold_min<ch)

    cleaned_detections_headmic_118 = [val[0][0]]
    for j in val[0][1:]:
        if j-cleaned_detections_headmic_118[-1] > 10000: # detections within 0.08 seconds are discareded to keep the first occurence of a click
            cleaned_detections_headmic_118.append(j)
    clicks_headmic_118 = np.append(clicks_headmic_118,np.array(cleaned_detections_headmic_118,dtype=np.uint64)+add_sample_amt)
    add_sample_amt+=len(ch)

20it [00:01, 15.86it/s]


In [222]:
# For channel headmic_35
add_sample_amt = 0
clicks_headmic_35 = np.array([],dtype=np.uint64)
for idx,file in tqdm.tqdm(enumerate(headmic_35_files)):
    sr,ch = wavfile.read(file)
    val = np.where(headmic_35_threshold_min<ch)

    cleaned_detections_headmic_35 = [val[0][0]]
    for j in val[0][1:]:
        if j-cleaned_detections_headmic_35[-1] > 10000: # detections within 0.08 seconds are discareded to keep the first occurence of a click
            cleaned_detections_headmic_35.append(j)
    clicks_headmic_35 = np.append(clicks_headmic_35,np.array(cleaned_detections_headmic_35,dtype=np.uint64)+add_sample_amt)
    add_sample_amt+=len(ch)

20it [00:01, 15.25it/s]


In [223]:
# matching ch_0 and headmic_118
headmic_118_matched_ch_0_clicks =  clicks_ch_0[np.abs(clicks_ch_0 - clicks_headmic_118[0]).argmin():]

In [ ]:
plt

array([  5012991,   5137995,   5263001, ..., 291460769, 291585776,
       291710779], dtype=uint64)

In [215]:
print(add_sample_amt/125000)

2398.633504


In [162]:
temp = headmic_118_matched_ch_0_clicks

In [170]:
temp_1 = headmic_118_matched_ch_0_clicks

In [179]:
temp_2 = headmic_118_matched_ch_0_clicks

In [182]:
def find_values_not_within_distance(arr1, arr2, distance):
    """
    Finds values in arr1 that are not within 'distance' of any value in arr2.

    Args:
        arr1 (np.ndarray): The first array to check elements from.
        arr2 (np.ndarray): The second array to compare against.
        distance (float or int): The maximum allowed distance.

    Returns:
        np.ndarray: An array containing the values from arr1 that are not
                    within 'distance' of any element in arr2.
    """
    not_within_range = []
    for val1 in arr1:
        # Calculate the absolute difference between val1 and all elements in arr2
        diffs = np.abs(arr2 - val1)
        # Check if any difference is within the specified distance
        if not np.any(diffs <= distance):
            not_within_range.append(val1)
    return np.array(not_within_range)

In [184]:
find_values_not_within_distance(temp,temp_2,1000)

array([11937169,  7960702], dtype=uint64)

In [195]:
11937169/125000

95.497352

In [196]:
95.497352/(60*2)

0.7958112666666667